# BioPAP Tutorial 1: Exploring Modules

Use this tutorial to explore BioPAP's processing modules:

- Test individual modules with your data
- Tweak configurations to see their effects
- Inspect outputs after each processing step
- Choose which modules you need for your preprocessing

### About the modules

When instantiated, all processing modules receive the data to be processed as a RawEDF-type object, which is assigned to an attribute labeled `raw`. Once done processing, the processed data is available as an attribute labeled `processed` of the module. The processed data has the same format as the raw input, so the output of one module can become the input of the next, thus allowing you to easily chain modules together.

The only exception is the Load module, which reads the initial data from disk without any further processing, and therefore only has raw data.

This notebook demonstrates some available modules and their basic configurations. For detailed information about each module, its parameters, and recommended usage patterns, consult the [API documentation](../docs/api/).


In [ ]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

In [ ]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne mne_bids

In [ ]:
from biopap.modules.anonymize import Anonymize
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.bids import BIDSConversion
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.downsample import Downsample
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.events_from_annotations import EventsFromAnnotations
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.load_data import LoadData
from biopap.modules.metadata import Metadata
from biopap.modules.notch_filter import NotchFilter
from biopap.modules.plot import Plot
from biopap.modules.rereference import Rereference
from biopap.modules.segmentation import Segmentation
from biopap.modules.save_output import SaveOutput
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

from biopap.modules.helpers import save_modules_config

from pathlib import Path
import mne

In [ ]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './data/my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = './data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists():
    raise FileNotFoundError(f"Config file {config_file} does not exist.")

if not Path(data_file).exists():
    raise FileNotFoundError(f"Data file {data_file} does not exist.")

## Test modules
### Loading data

In [ ]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file)

In [ ]:
# Display basic info
load.raw.info

In [ ]:
# Show channel info
load.raw.describe()

### Anonymization

In [ ]:
# Anonymize the data: removes personal identifiers and shifts recording date
# - 'daysback' defines the number of days subtracted from the original measurement date in the data. Its value can either be a 
#   positive integer, or the word 'random', in which case the program will subtract a random number between 1 and 100.
# - 'experimenter' and 'participant_id' are both optional.

ano = Anonymize(raw=load.raw, daysback=100, experimenter='J. Jansen', participant_id='AB00123')
ano.processed.info

### Rereferencing

In [ ]:
ref = Rereference(raw=ano.processed, ref_channels=['EXG1', 'EXG2'])
ref.processed.info

### Create EOG channels

In [ ]:
# Create horizontal and vertical EOG channels from electrode pairs
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

eog.processed.info

### Set Channel Types and Montage

In [ ]:
# Define channel types mapping the original channel names to MNE types
# Adjust the channel names and montage based on your actual data
channel_types = {
        'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
        'gsr': ['GSR1', 'GSR2'],
        'resp': ['Resp'],
        'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)
montage.processed.info

In [ ]:
# See available montages with:
mne.channels.get_builtin_montages()

### BIDS Conversion

In [ ]:
# (OPTIONAL) Convert raw data to BIDS (Brain Imaging Data Structure) format
# BIDS conversion should be done on the raw, unprocessed data to preserve original parameters
# You'll need to prepare an events file with your marker definitions
# The events file should be a CSV with columns: ID and label

# Example of BIDS conversion (comment out if not needed)
# bids_convert = BIDSConversion(
#     raw=load.raw,  # Use the original raw data, NOT processed data
#     eventsfile='./data/events_metadata.csv',  # Path to your events/markers file
#     bids_root='./bids',  # Root directory for BIDS dataset
#     subject='001',  # Subject identifier
#     session='01',  # Session identifier
#     task='default_task',  # Task name
#     run=1,  # Run number
#     overwrite=False,
# )
# print("Data converted and saved to BIDS format")
# print(f"BIDS root: ./bids")


### Filtering (IIR Filter)

In [ ]:
# Apply IIR band-pass filter to remove low and high frequency noise
flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

# IIRFilter has two additional paramaters:"
# - order (defaults to 4)
# - ftype (which defautts to 'butter')

In [ ]:
# Optionally, save a plot of the current state by specifying save folder. The screenshot will take the basename of the original
# source file, optionally taking a postfix to distinguish between multiple screenshots on separate points in the pipeline.
# This step is not interactive, so can also be used in a pipeline intended to run unattended.
# N.B., showing the interactive plot and saving cannot be done at the same time, but you can add multiple subsequent Plot-steps.
_ = Plot(raw=flt.processed, save_folder='./', save_postfix='IRR')

### Downsampling

In [ ]:
# Downsample the data to reduce file size and computational requirements
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

# Additional parameter:
# - resample_events (default: True)

dwn.processed.info

### Blink detection

In [ ]:
# Automatically detect blinks from EOG channels
blink = BlinkDetection(raw=dwn.processed)

# Use the Plot-module see the blink annotations
_ = Plot(raw=blink.processed)

### Segmentation (Epoching)

In [ ]:
# Create epochs around events/markers detected on the stimulation channel
# Adjust these parameters based on your task and stimulus markers
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'}
)

seg.processed

### Artifact Rejection

In [ ]:
# Reject epochs with defined limits
# This identifies bad channels based on voltage criteria and either interpolates or replaces them with NaNs
artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100
)

### Baseline Correction

In [ ]:
# Apply baseline correction to epochs
# The baseline period is specified as (start, end) in seconds relative to event
baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0)
)


### Visualization with Plot

In [ ]:
# Visualize the data before further processing or saving
plt = Plot(raw=baseline.processed, title="Preprocessed data")

### Saving Output

In [ ]:
# Save the processed data in various formats
# FIF format (MNE's native format)
save_output = SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    overwrite=True
)

---

## Save module settings to a configuration file

After testing modules and choosing their settings, use `save_modules_config` to write the selected modules' configurations to a configuration file.

The order of the modules is important, as this will also be the order in which the modules will be added when a new pipeline is created from the saved configuration file.

Note that you only need to pass the function a list of the modules that you actually want added to the config. 

Also note that it is not necessary to save the settings from the `Load` module, as that module is also integrated in the `Pipeline` class.

### About Configuration Files

Configuration files store all module parameters in one place. This is useful for **batch processing**, to have the same set of parameters to process dataset identically. 

For more about configuration files, see the [configuration tutorial](../docs/tutorials/configuration.md).

In [ ]:
# define configuration file name
cfg_file = './config.yaml'

# write module configurations to configuration file
save_modules_config(
    modules=[ano, ref, eog, montage, flt, dwn, seg, artifact, baseline, plt, save_output], 
    config_path=cfg_file
)